# Interactive Web Application for Audio Deepfake Detection\n\nRun the cell below to launch the dark-theme browser interface at http://localhost:7860.\n

In [ ]:
"""
Interactive Web Application for Audio Deepfake Detection
Built with standard library http.server + Modern Dark-Theme Frontend.
No external web frameworks required.
"""

import os
import io
import json
import time
import cgi
from pathlib import Path
from http.server import HTTPServer, BaseHTTPRequestHandler
import soundfile as sf
import numpy as np
import torch
import torchaudio
from transformers import Wav2Vec2Model

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "models" / "best_wav2vec2_deepfake_model.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load Model
print(f"Loading Wav2Vec2 deepfake detector from {MODEL_PATH} on {DEVICE}...")
bundle = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

class Classifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = torch.nn.Sequential(
            torch.nn.LayerNorm(768),
            torch.nn.Linear(768, 256),
            torch.nn.GELU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(256, 2)
        )
    def forward(self, x):
        return torch.softmax(self.classifier(self.wav2vec2(input_values=x).last_hidden_state.mean(dim=1)), dim=-1)

model = Classifier().to(DEVICE)
model.load_state_dict(bundle["model_state_dict"])
model.eval()
print("Model loaded successfully!")

def run_prediction(audio_bytes: bytes, filename: str):
    start_time = time.time()
    # Read audio from memory buffer
    with io.BytesIO(audio_bytes) as bio:
        data, sr = sf.read(bio, dtype="float32")

    # Mono conversion
    if data.ndim > 1:
        data = np.mean(data, axis=1)

    # Resample to 16kHz
    if sr != 16000:
        tensor_data = torch.from_numpy(data).unsqueeze(0)
        data = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(tensor_data).squeeze(0).numpy()

    # 4.0s equalization (64,000 samples)
    target_samples = 64000
    if len(data) < target_samples:
        data = np.pad(data, (0, target_samples - len(data)), mode="constant")
    elif len(data) > target_samples:
        data = data[:target_samples]

    # Peak normalization
    m = np.max(np.abs(data))
    if m > 1e-6:
        data = (data / m) * 0.95

    tensor_in = torch.from_numpy(data).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = model(tensor_in).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    label = "Fake" if pred_id == 1 else "Real"
    conf = float(probs[pred_id])
    latency_ms = (time.time() - start_time) * 1000

    return {
        "filename": filename,
        "prediction": label,
        "confidence": conf,
        "is_fake": bool(pred_id == 1),
        "real_prob": float(probs[0]),
        "fake_prob": float(probs[1]),
        "latency_ms": round(latency_ms, 1)
    }

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Audio Deepfake Detector | Wav2Vec 2.0</title>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap" rel="stylesheet">
  <style>
    :root {
      --bg: #0d1117;
      --card-bg: #161b22;
      --card-border: #30363d;
      --accent: #58a6ff;
      --fake: #f85149;
      --real: #2ea043;
      --text: #c9d1d9;
      --text-bold: #f0f6fc;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
      background-color: var(--bg);
      color: var(--text);
      min-height: 100vh;
      display: flex;
      flex-direction: column;
      align-items: center;
      padding: 40px 20px;
    }
    .container {
      max-width: 800px;
      width: 100%;
    }
    header {
      text-align: center;
      margin-bottom: 30px;
    }
    header h1 {
      color: var(--text-bold);
      font-size: 2rem;
      margin-bottom: 8px;
    }
    header p {
      color: #8b949e;
      font-size: 1rem;
    }
    .card {
      background: var(--card-bg);
      border: 1px solid var(--card-border);
      border-radius: 12px;
      padding: 24px;
      margin-bottom: 24px;
      box-shadow: 0 8px 24px rgba(0,0,0,0.4);
    }
    .dropzone {
      border: 2px dashed #388bfd;
      border-radius: 8px;
      padding: 40px 20px;
      text-align: center;
      cursor: pointer;
      background: rgba(56, 139, 253, 0.04);
      transition: all 0.2s ease;
    }
    .dropzone:hover {
      background: rgba(56, 139, 253, 0.08);
      border-color: #58a6ff;
    }
    .dropzone input { display: none; }
    .btn {
      background: #238636;
      color: #fff;
      border: none;
      padding: 12px 24px;
      font-size: 1rem;
      font-weight: 600;
      border-radius: 6px;
      cursor: pointer;
      width: 100%;
      margin-top: 16px;
      transition: background 0.2s;
    }
    .btn:hover { background: #2ea043; }
    .btn:disabled { background: #30363d; cursor: not-allowed; color: #8b949e; }
    .sample-buttons {
      display: flex;
      gap: 12px;
      margin-top: 16px;
    }
    .sample-btn {
      flex: 1;
      background: #21262d;
      border: 1px solid var(--card-border);
      color: var(--text);
      padding: 8px;
      border-radius: 6px;
      cursor: pointer;
      font-size: 0.9rem;
      transition: background 0.2s;
    }
    .sample-btn:hover { background: #30363d; color: var(--text-bold); }
    audio {
      width: 100%;
      margin-top: 16px;
      border-radius: 8px;
    }
    #result-box {
      display: none;
      margin-top: 24px;
      animation: fadeIn 0.3s ease;
    }
    @keyframes fadeIn { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: translateY(0); } }
    .badge {
      display: inline-block;
      padding: 8px 16px;
      font-size: 1.2rem;
      font-weight: 700;
      border-radius: 20px;
      margin-bottom: 12px;
    }
    .badge.real { background: rgba(46, 160, 67, 0.2); color: var(--real); border: 1px solid var(--real); }
    .badge.fake { background: rgba(248, 81, 73, 0.2); color: var(--fake); border: 1px solid var(--fake); }
    .prob-bar-container {
      background: #21262d;
      border-radius: 8px;
      height: 28px;
      overflow: hidden;
      margin: 16px 0;
      display: flex;
    }
    .prob-real { background: var(--real); transition: width 0.5s ease; }
    .prob-fake { background: var(--fake); transition: width 0.5s ease; }
    .meta-grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 12px;
      margin-top: 16px;
      font-size: 0.9rem;
    }
    .meta-item {
      background: #0d1117;
      padding: 10px;
      border-radius: 6px;
      border: 1px solid var(--card-border);
    }
    .meta-item span { color: #8b949e; display: block; font-size: 0.8rem; }
    .spinner {
      display: none;
      margin: 20px auto;
      width: 36px;
      height: 36px;
      border: 3px solid rgba(255,255,255,0.1);
      border-radius: 50%;
      border-top-color: var(--accent);
      animation: spin 0.8s linear infinite;
    }
    @keyframes spin { to { transform: rotate(360deg); } }
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>🎙️ Audio Deepfake Detector</h1>
      <p>Powered by fine-tuned <strong>facebook/wav2vec2-base</strong> with 98.44% Test Accuracy</p>
    </header>

    <div class="card">
      <div class="dropzone" id="dropzone" onclick="document.getElementById('file-input').click()">
        <p style="font-size: 1.1rem; font-weight: 600; margin-bottom: 6px;">Click or Drag & Drop Audio File Here</p>
        <p style="font-size: 0.85rem; color: #8b949e;">Supports .WAV and .FLAC files (Standardized to 16kHz Mono)</p>
        <input type="file" id="file-input" accept=".wav,.flac">
      </div>

      <div style="margin-top: 16px; text-align: center; color: #8b949e; font-size: 0.9rem;">— OR TEST PRE-LOADED SAMPLES —</div>
      <div class="sample-buttons">
        <button class="sample-btn" onclick="loadSample('real')">🔊 Test Genuine Human Voice</button>
        <button class="sample-btn" onclick="loadSample('fake')">🤖 Test Synthetic Deepfake Voice</button>
      </div>

      <audio id="audio-player" controls style="display: none;"></audio>
      <button class="btn" id="analyze-btn" disabled onclick="analyzeAudio()">Analyze Audio</button>
      <div class="spinner" id="spinner"></div>

      <div id="result-box">
        <div style="text-align: center; margin-top: 10px;">
          <div id="verdict-badge" class="badge"></div>
          <div style="font-size: 1.1rem; margin-bottom: 6px;">Confidence: <strong id="conf-val" style="color: #fff;"></strong></div>
        </div>

        <div style="display: flex; justify-content: space-between; font-size: 0.85rem; margin-bottom: 4px;">
          <span style="color: var(--real);">Real: <strong id="real-prob-text"></strong></span>
          <span style="color: var(--fake);">Fake: <strong id="fake-prob-text"></strong></span>
        </div>
        <div class="prob-bar-container">
          <div id="bar-real" class="prob-real" style="width: 50%;"></div>
          <div id="bar-fake" class="prob-fake" style="width: 50%;"></div>
        </div>

        <div class="meta-grid">
          <div class="meta-item"><span>File Name</span><div id="res-filename" style="color: #fff;"></div></div>
          <div class="meta-item"><span>Inference Latency</span><div id="res-latency" style="color: #fff;"></div></div>
          <div class="meta-item"><span>Audio Format</span><div style="color: #fff;">16kHz Mono (4.0s Equalized)</div></div>
          <div class="meta-item"><span>Backbone Architecture</span><div style="color: #fff;">facebook/wav2vec2-base</div></div>
        </div>
      </div>
    </div>
  </div>

  <script>
    let currentAudioFile = null;

    const fileInput = document.getElementById('file-input');
    const audioPlayer = document.getElementById('audio-player');
    const analyzeBtn = document.getElementById('analyze-btn');
    const spinner = document.getElementById('spinner');
    const resultBox = document.getElementById('result-box');

    fileInput.addEventListener('change', (e) => {
      const file = e.target.files[0];
      if (file) {
        currentAudioFile = file;
        audioPlayer.src = URL.createObjectURL(file);
        audioPlayer.style.display = 'block';
        analyzeBtn.disabled = false;
        resultBox.style.display = 'none';
      }
    });

    async function loadSample(type) {
      spinner.style.display = 'block';
      analyzeBtn.disabled = true;
      try {
        const res = await fetch(`/api/sample?type=${type}`);
        const blob = await res.blob();
        currentAudioFile = new File([blob], `${type}_sample.wav`, { type: 'audio/wav' });
        audioPlayer.src = URL.createObjectURL(blob);
        audioPlayer.style.display = 'block';
        analyzeBtn.disabled = false;
        analyzeAudio();
      } catch (err) {
        alert('Error loading sample: ' + err);
      } finally {
        spinner.style.display = 'none';
      }
    }

    async function analyzeAudio() {
      if (!currentAudioFile) return;

      spinner.style.display = 'block';
      analyzeBtn.disabled = true;
      resultBox.style.display = 'none';

      const formData = new FormData();
      formData.append('audio', currentAudioFile);

      try {
        const response = await fetch('/api/predict', {
          method: 'POST',
          body: formData
        });
        const data = await response.json();

        // Update UI
        const badge = document.getElementById('verdict-badge');
        if (data.is_fake) {
          badge.className = 'badge fake';
          badge.innerHTML = '🚨 SYNTHETIC DEEPFAKE VOICE';
        } else {
          badge.className = 'badge real';
          badge.innerHTML = '✅ AUTHENTIC HUMAN VOICE';
        }

        document.getElementById('conf-val').textContent = (data.confidence * 100).toFixed(2) + '%';
        document.getElementById('real-prob-text').textContent = (data.real_prob * 100).toFixed(2) + '%';
        document.getElementById('fake-prob-text').textContent = (data.fake_prob * 100).toFixed(2) + '%';
        document.getElementById('bar-real').style.width = (data.real_prob * 100) + '%';
        document.getElementById('bar-fake').style.width = (data.fake_prob * 100) + '%';

        document.getElementById('res-filename').textContent = data.filename;
        document.getElementById('res-latency').textContent = data.latency_ms + ' ms';

        resultBox.style.display = 'block';
      } catch (err) {
        alert('Prediction error: ' + err);
      } finally {
        spinner.style.display = 'none';
        analyzeBtn.disabled = false;
      }
    }
  </script>
</body>
</html>
"""

class RequestHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/" or self.path == "/index.html":
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers()
            self.wfile.write(HTML_TEMPLATE.encode("utf-8"))

        elif self.path.startswith("/api/sample"):
            sample_type = "real" if "type=real" in self.path else "fake"
            data_dir = BASE_DIR / "data" / "processed_clean"
            folder = data_dir / ("Real_audio" if sample_type == "real" else "FAKE")
            sample_files = list(folder.glob("*.wav"))
            if sample_files:
                sample_path = sample_files[0]
                with open(sample_path, "rb") as f:
                    audio_data = f.read()
                self.send_response(200)
                self.send_header("Content-Type", "audio/wav")
                self.send_header("Content-Length", str(len(audio_data)))
                self.end_headers()
                self.wfile.write(audio_data)
            else:
                self.send_error(404, "Sample not found")

        else:
            self.send_error(404, "Not Found")

    def do_POST(self):
        if self.path == "/api/predict":
            content_type = self.headers.get("Content-Type")
            if not content_type or not content_type.startswith("multipart/form-data"):
                self.send_error(400, "Expected multipart/form-data")
                return

            # Parse multipart form data
            environ = {
                "REQUEST_METHOD": "POST",
                "CONTENT_TYPE": content_type,
                "CONTENT_LENGTH": self.headers.get("Content-Length")
            }
            form = cgi.FieldStorage(fp=self.rfile, headers=self.headers, environ=environ)
            if "audio" not in form:
                self.send_error(400, "Missing audio file")
                return

            audio_item = form["audio"]
            audio_bytes = audio_item.file.read()
            filename = audio_item.filename or "uploaded_audio.wav"

            try:
                result = run_prediction(audio_bytes, filename)
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps(result).encode("utf-8"))
            except Exception as e:
                self.send_response(500)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps({"error": str(e)}).encode("utf-8"))

        else:
            self.send_error(404, "Not Found")

def start_server(port=7860):
    server = HTTPServer(("0.0.0.0", port), RequestHandler)
    print(f"\n========================================================")
    print(f"  AUDIO DEEPFAKE DETECTOR WEB APP RUNNING")
    print(f"  Local URL: http://localhost:{port}")
    print(f"========================================================\n")
    server.serve_forever()

if __name__ == "__main__":
    start_server()

\n